# LangGraph Email Triage Agent

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shanjai-raj/commune-cookbook/blob/main/notebooks/langgraph_email_agent.ipynb)

LangChain Chains execute linearly: input goes in, output comes out. That model works well for a single question-answer cycle, but email triage is fundamentally a **routing problem** — the action taken depends on the classification result, and different email types require different processing paths.

LangGraph adds three things that Chains cannot provide:

- **Conditional edges** — after classifying an email, the graph routes to `reply`, `escalate`, or `archive` based on the result. Chains would need nested if-statements or multiple separate chains.
- **State persistence via checkpointing** — the graph can be interrupted (e.g., waiting for human approval on urgent cases) and resumed without losing state. A Chain has no built-in persistence.
- **Fault tolerance** — if a node fails mid-graph, checkpointing allows resuming from the last successful node rather than restarting from scratch.

This notebook builds a 3-node email triage graph: `classify_email` → route → [`compose_reply` | `escalate_via_sms` | `archive_email`].

**Prerequisites:**
- [Commune API key](https://commune.email) (free tier)
- [OpenAI API key](https://platform.openai.com)
- `pip install commune-mail langgraph langchain-openai openai`

In [1]:
!pip install commune-mail langgraph langchain-openai openai -q

import os
import json
from typing import Optional, Literal
from typing_extensions import TypedDict

from commune import CommuneClient
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

COMMUNE_API_KEY = os.environ.get("COMMUNE_API_KEY", "comm_your_key_here")
OPENAI_API_KEY  = os.environ.get("OPENAI_API_KEY",  "sk-your_key_here")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

client = CommuneClient(api_key=COMMUNE_API_KEY)
llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0)

SUPPORT_INBOX_ID = os.environ.get("SUPPORT_INBOX_ID", "inbox_sup_01")
SMS_FROM_NUMBER  = os.environ.get("SMS_FROM_NUMBER",  "+15550001234")
ON_CALL_NUMBER   = os.environ.get("ON_CALL_NUMBER",   "+15559876543")

print("✓ Dependencies installed")
print("✓ Commune client ready (sync)")
print("✓ OpenAI key set")

✓ Dependencies installed
✓ Commune client ready (sync)
✓ OpenAI key set


## The 3-Node Triage Graph

The graph has one entry node and three terminal nodes:

```
classify_email
      |
  route_email  (conditional edge)
  /     |     \
reply  escalate  archive
```

- **`classify_email`** — reads the email body and calls the LLM to assign one of four labels: `support`, `lead`, `spam`, `urgent`.
- **`route_email`** — a conditional edge function that returns the name of the next node based on `state["classification"]`. Not a node itself — LangGraph calls it between nodes.
- **`compose_reply`** — for `support` and `lead` emails. Generates a reply and sends it via Commune, passing `thread_id` for continuity.
- **`escalate_via_sms`** — for `urgent` emails. Sends an SMS alert via `client.sms.send()` so the on-call human is notified immediately.
- **`archive_email`** — for `spam`. Sends a minimal acknowledgment (or nothing) and marks the thread complete.

State flows through every node as a single `EmailState` TypedDict, so each node can read and write fields without passing arguments explicitly.

In [2]:
class EmailState(TypedDict):
    """Shared state passed through every node in the triage graph.

    All fields are optional at construction time; nodes populate them
    as the graph executes. thread_id is critical for reply continuity.
    """
    # Inbound email fields (populated by the webhook caller)
    thread_id:      Optional[str]   # Commune thread_id from the inbound webhook
    inbox_id:       str             # Commune inbox_id to send from
    sender:         str             # Sender email address
    subject:        str             # Email subject
    body:           str             # Plain-text email body

    # Set by classify_email node
    classification: Optional[str]   # 'support' | 'lead' | 'spam' | 'urgent'

    # Set by terminal nodes
    reply_sent:     bool            # True if compose_reply sent a message
    sms_sent:       bool            # True if escalate_via_sms fired
    error:          Optional[str]   # Error message if a node failed


print("✓ EmailState TypedDict defined")
print("Fields: thread_id, inbox_id, sender, subject, body, classification, reply_sent, sms_sent, error")

✓ EmailState TypedDict defined
Fields: thread_id, inbox_id, sender, subject, body, classification, reply_sent, sms_sent, error


In [3]:
CLASSIFY_PROMPT = """You are an email classifier. Given an email subject and body, classify it into exactly one of these categories:

- support    : Bug reports, how-to questions, technical issues
- lead       : Sales inquiries, pricing questions, demo requests
- spam       : Unsolicited marketing, irrelevant content, phishing
- urgent     : System down, data loss, security breach, SLA breach

Reply with ONLY the category name in lowercase. No explanation."""


def classify_email(state: EmailState) -> EmailState:
    """Node: classify the inbound email using the LLM.

    Reads state['subject'] and state['body'].
    Writes state['classification'].

    Returns the updated state dict (LangGraph merges it into the graph state).
    """
    response = llm.invoke([
        SystemMessage(content=CLASSIFY_PROMPT),
        HumanMessage(content=f"Subject: {state['subject']}\n\nBody: {state['body']}"),
    ])

    raw = response.content.strip().lower()
    # Clamp to valid categories in case the LLM drifts
    classification = raw if raw in ("support", "lead", "spam", "urgent") else "support"

    return {"classification": classification}


# Demo (without LLM call)
demo_state: EmailState = {
    "thread_id": None,
    "inbox_id": SUPPORT_INBOX_ID,
    "sender": "alice@example.com",
    "subject": "App keeps crashing on upload",
    "body": "Every time I try to upload a file over 10MB the app crashes.",
    "classification": None,
    "reply_sent": False,
    "sms_sent": False,
    "error": None,
}

print("✓ classify_email node defined")
print(f"Demo classification for '{demo_state['subject']}':\n  classification = support")

✓ classify_email node defined
Demo classification for 'App keeps crashing on upload':
  classification = support


In [4]:
def route_email(state: EmailState) -> Literal["reply", "escalate", "archive"]:
    """Conditional edge function: map classification to next node name.

    LangGraph calls this function after classify_email completes.
    The return value must be a string matching one of the add_conditional_edges targets.

    Returns:
        'reply'    for support and lead emails (handled by compose_reply node)
        'escalate' for urgent emails (handled by escalate_via_sms node)
        'archive'  for spam (handled by archive_email node)
    """
    c = state.get("classification", "support")
    if c in ("support", "lead"):
        return "reply"
    if c == "urgent":
        return "escalate"
    return "archive"  # spam or unknown


# Verify routing logic
print("✓ route_email conditional edge defined")
print("Routing tests:")
for cls in ("support", "lead", "spam", "urgent"):
    test = {**demo_state, "classification": cls}
    print(f"  {cls:<8} -> {route_email(test)}")

✓ route_email conditional edge defined
Routing tests:
  support  -> reply
  lead     -> reply
  spam     -> archive
  urgent   -> escalate


In [5]:
REPLY_PROMPT = """You are a professional customer support specialist. Given an inbound customer email, write a helpful, concise reply (under 120 words). Be warm but efficient. Sign off as 'The Support Team'."""


def compose_reply(state: EmailState) -> EmailState:
    """Node: generate and send a reply for support and lead emails.

    Reads state['body'], state['subject'], state['thread_id'].
    Passes thread_id to messages.send() to maintain conversation continuity.
    Writes state['reply_sent'].
    """
    response = llm.invoke([
        SystemMessage(content=REPLY_PROMPT),
        HumanMessage(
            content=f"From: {state['sender']}\nSubject: {state['subject']}\n\n{state['body']}"
        ),
    ])

    reply_text = response.content.strip()

    # CRITICAL: pass thread_id so the reply lands in the existing conversation.
    # If thread_id is None (first contact), Commune creates a new thread and
    # returns a thread_id we can store for future replies.
    result = client.messages.send(
        to=state["sender"],
        subject=f"Re: {state['subject']}",
        text=reply_text,
        inbox_id=state["inbox_id"],
        thread_id=state["thread_id"],  # None -> new thread; set -> reply in thread
    )

    return {"reply_sent": True, "thread_id": result.thread_id}


print("✓ compose_reply node defined")
print("Demo (simulated \u2014 no API call):")
print(f"  Would send reply to {demo_state['sender']}")
print(f"  thread_id passed to send(): {demo_state['thread_id']}  (new thread on first contact)")
print("  reply_sent = True")

✓ compose_reply node defined
Demo (simulated — no API call):
  Would send reply to alice@example.com
  thread_id passed to send(): None  (new thread on first contact)
  reply_sent = True


In [6]:
def escalate_via_sms(state: EmailState) -> EmailState:
    """Node: alert the on-call team via SMS for urgent emails.

    Uses client.sms.send() to notify the on-call number immediately.
    Also sends an auto-acknowledgment email to the customer.

    Reads state['sender'], state['subject'], state['thread_id'], state['inbox_id'].
    Writes state['sms_sent'], state['reply_sent'].
    """
    # 1. Alert on-call team via SMS
    sms_body = (
        f"[URGENT EMAIL] From: {state['sender']} | "
        f"Subject: {state['subject']} | "
        f"Needs immediate attention."
    )

    sms_result = client.sms.send(
        to=ON_CALL_NUMBER,
        body=sms_body,
    )

    # 2. Send auto-acknowledgment to the customer so they know we received it
    ack_result = client.messages.send(
        to=state["sender"],
        subject=f"Re: {state['subject']}",
        text=(
            "We received your message and have alerted our on-call team. "
            "You will hear from us within 15 minutes."
        ),
        inbox_id=state["inbox_id"],
        thread_id=state["thread_id"],
    )

    return {"sms_sent": True, "reply_sent": True, "thread_id": ack_result.thread_id}


print("✓ escalate_via_sms node defined")
print("Demo SMS that would be sent for urgent classification:")
print(f"  To: {ON_CALL_NUMBER}")
print(
    "  Body: [URGENT EMAIL] From: alice@example.com | "
    "Subject: Production database unreachable | Needs immediate attention."
)

✓ escalate_via_sms node defined
Demo SMS that would be sent for urgent classification:
  To: +15559876543
  Body: [URGENT EMAIL] From: alice@example.com | Subject: Production database unreachable | Needs immediate attention.


In [7]:
def archive_email(state: EmailState) -> EmailState:
    """Node: handle spam and low-priority emails.

    For spam, we do not send a reply (replying to spam confirms the
    address is active and invites more spam). We simply mark the
    processing as complete without sending anything.

    In a production system you would write the thread_id to a
    spam-tracking table for audit purposes.

    Writes state['reply_sent'] = False (already the default).
    """
    # Log for audit trail (replace with DB write in production)
    print(f"  [archive] Spam/low-priority email from {state['sender']} — no reply sent.")

    return {"reply_sent": False}


print("✓ archive_email node defined")
print("Spam emails: no reply sent, reply_sent = False")

✓ archive_email node defined
Spam emails: no reply sent, reply_sent = False


In [8]:
# Build the LangGraph state machine

checkpointer = MemorySaver()  # In-memory for dev; use SqliteSaver or RedisSaver in prod

builder = StateGraph(EmailState)

# Add nodes
builder.add_node("classify_email",  classify_email)
builder.add_node("compose_reply",   compose_reply)
builder.add_node("escalate_via_sms", escalate_via_sms)
builder.add_node("archive_email",   archive_email)

# Entry point
builder.set_entry_point("classify_email")

# Conditional edge: route_email returns the node name to visit next
builder.add_conditional_edges(
    source="classify_email",
    path=route_email,
    path_map={
        "reply":    "compose_reply",
        "escalate": "escalate_via_sms",
        "archive":  "archive_email",
    },
)

# All terminal nodes go to END
builder.add_edge("compose_reply",    END)
builder.add_edge("escalate_via_sms", END)
builder.add_edge("archive_email",    END)

# Compile with checkpointer so graph state is persisted between invocations
graph = builder.compile(checkpointer=checkpointer)

print("✓ Graph compiled with MemorySaver checkpointer")
print("Nodes: classify_email, compose_reply, escalate_via_sms, archive_email")
print("Entry point: classify_email")
print("Conditional edges from classify_email:")
print("  support/lead -> reply (compose_reply)")
print("  urgent       -> escalate (escalate_via_sms)")
print("  spam         -> archive (archive_email)")

✓ Graph compiled with MemorySaver checkpointer
Nodes: classify_email, compose_reply, escalate_via_sms, archive_email
Entry point: classify_email
Conditional edges from classify_email:
  support/lead -> reply (compose_reply)
  urgent       -> escalate (escalate_via_sms)
  spam         -> archive (archive_email)


## Contrastive: WRONG vs RIGHT Thread Continuity in State

The single most common mistake when building LangGraph email agents is omitting `thread_id` from the state definition. Without it, the `compose_reply` node cannot pass `thread_id` to `messages.send()`, and every reply creates a **new** email thread instead of continuing the existing conversation.

The symptom is subtle: the agent sends replies that appear correct in isolation, but the customer receives disconnected emails instead of a single thread. Support agents lose conversation history. The customer has to re-explain their issue on every reply.

**WRONG:** `thread_id` not in state, so `compose_reply` cannot read it.

**RIGHT:** `thread_id` in `EmailState`, seeded from the webhook payload, passed to every `messages.send()` call.

In [9]:
# WRONG: thread_id missing from state

class WrongEmailState(TypedDict):
    """Missing thread_id — every reply breaks thread continuity."""
    inbox_id:       str
    sender:         str
    subject:        str
    body:           str
    classification: Optional[str]
    reply_sent:     bool
    # BUG: thread_id is not here — cannot be passed from webhook to reply node


def compose_reply_wrong(state: WrongEmailState) -> WrongEmailState:
    """BUG: cannot access thread_id because it is not in WrongEmailState."""
    reply_text = "Thank you for reaching out. We are looking into your issue."

    # thread_id=None always — creates a NEW thread on every call
    result = client.messages.send(
        to=state["sender"],
        subject=f"Re: {state['subject']}",
        text=reply_text,
        inbox_id=state["inbox_id"],
        thread_id=None,  # BUG: always None — disconnected reply
    )
    return {"reply_sent": True}


print("--- WRONG: thread_id not in state ---")
print()
print("Incoming webhook: thread_id = thr_abc123")
print("WrongEmailState has no thread_id field.")
print("compose_reply_wrong(): thread_id passed to send() = None")
print()
print("Result: Commune creates a NEW thread instead of replying.")
print("Customer receives email #2 in a separate thread.")
print("Support agent cannot see prior messages. Thread is fragmented.")

--- WRONG: thread_id not in state ---

Incoming webhook: thread_id = thr_abc123
WrongEmailState has no thread_id field.
compose_reply_wrong(): thread_id passed to send() = None

Result: Commune creates a NEW thread instead of replying.
Customer receives email #2 in a separate thread.
Support agent cannot see prior messages. Thread is fragmented.


In [10]:
# RIGHT: thread_id in EmailState, seeded from webhook, passed to send()

def compose_reply_correct(state: EmailState) -> EmailState:
    """Correct: reads thread_id from state and passes it to messages.send()."""
    reply_text = "Thank you for reaching out. We are looking into your issue."

    # thread_id comes from the inbound webhook payload via EmailState
    result = client.messages.send(
        to=state["sender"],
        subject=f"Re: {state['subject']}",
        text=reply_text,
        inbox_id=state["inbox_id"],
        thread_id=state["thread_id"],  # CORRECT: continues the existing conversation
    )
    # Write the (possibly new) thread_id back to state for subsequent nodes
    return {"reply_sent": True, "thread_id": result.thread_id}


# Simulate how the webhook caller populates state
webhook_payload = {
    "sender": "alice@example.com",
    "subject": "App keeps crashing",
    "text": "Files over 10MB cause a crash.",
    "thread_id": "thr_abc123",  # from the inbound Commune webhook payload
    "inbox_id": SUPPORT_INBOX_ID,
}

initial_state: EmailState = {
    "thread_id":      webhook_payload["thread_id"],   # seeded here
    "inbox_id":       webhook_payload["inbox_id"],
    "sender":         webhook_payload["sender"],
    "subject":        webhook_payload["subject"],
    "body":           webhook_payload["text"],
    "classification": None,
    "reply_sent":     False,
    "sms_sent":       False,
    "error":          None,
}

print("--- RIGHT: thread_id in EmailState, propagated correctly ---")
print()
print(f"Incoming webhook: thread_id = {webhook_payload['thread_id']}")
print(f"EmailState.thread_id = {initial_state['thread_id']}")
print(f"compose_reply(): thread_id passed to send() = {initial_state['thread_id']}")
print()
print("Result: Reply lands in the existing thread.")
print("Customer sees a single continuous conversation.")
print("Support agent has full message history.")
print()
print("Key difference:")
print("  WRONG: WrongEmailState has no thread_id field -> send(thread_id=None) always")
print(f"  RIGHT: EmailState.thread_id = {initial_state['thread_id']}    -> send(thread_id='{initial_state['thread_id']}')")

--- RIGHT: thread_id in EmailState, propagated correctly ---

Incoming webhook: thread_id = thr_abc123
EmailState.thread_id = thr_abc123
compose_reply(): thread_id passed to send() = thr_abc123

Result: Reply lands in the existing thread.
Customer sees a single continuous conversation.
Support agent has full message history.

Key difference:
  WRONG: WrongEmailState has no thread_id field -> send(thread_id=None) always
  RIGHT: EmailState.thread_id = thr_abc123    -> send(thread_id='thr_abc123')


In [11]:
# Simulated full graph execution trace with a support email
# (Replace with graph.invoke(initial_state, config) when running with real API keys)

print("=== Graph execution trace (simulated) ===")
print()
print("Input state:")
print("  sender:   alice@example.com")
print("  subject:  App keeps crashing on upload")
print("  thread_id: None (first contact)")
print()
print("Node 1/2: classify_email")
print("  LLM input:  Subject: App keeps crashing on upload | Body: Every time I try to upload...")
print("  LLM output: 'support'")
print("  State after: classification = support")
print()
print("Conditional edge: route_email('support') -> 'reply' -> compose_reply")
print()
print("Node 2/2: compose_reply")
print("  thread_id read from state: None")
print("  messages.send(to='alice@example.com', thread_id=None, inbox_id='inbox_sup_01')")
print("  API response: SendMessageResult(message_id='msg_r7k2p9', thread_id='thr_newx01', status='sent')")
print("  State after: reply_sent = True, thread_id = thr_newx01")
print()
print("Graph complete. Final state:")
print("  classification: support")
print("  reply_sent:     True")
print("  sms_sent:       False")
print("  thread_id:      thr_newx01")
print("  error:          None")

=== Graph execution trace (simulated) ===

Input state:
  sender:   alice@example.com
  subject:  App keeps crashing on upload
  thread_id: None (first contact)

Node 1/2: classify_email
  LLM input:  Subject: App keeps crashing on upload | Body: Every time I try to upload...
  LLM output: 'support'
  State after: classification = support

Conditional edge: route_email('support') -> 'reply' -> compose_reply

Node 2/2: compose_reply
  thread_id read from state: None
  messages.send(to='alice@example.com', thread_id=None, inbox_id='inbox_sup_01')
  API response: SendMessageResult(message_id='msg_r7k2p9', thread_id='thr_newx01', status='sent')
  State after: reply_sent = True, thread_id = thr_newx01

Graph complete. Final state:
  classification: support
  reply_sent:     True
  sms_sent:       False
  thread_id:      thr_newx01
  error:          None


In [12]:
# Checkpointer pattern: resume after crash without re-running completed nodes
# MemorySaver is used here; in production use SqliteSaver or RedisSaver.

# The graph thread_id here is LangGraph's run identifier,
# distinct from Commune's email thread_id.

graph_run_config = {"configurable": {"thread_id": "email-001"}}

# First run: simulate classify_email completing but crashing before compose_reply
print("=== Checkpointer: resuming after crash ===")
print()
print("Run 1: classify_email completed. Crash before compose_reply.")
print("  Checkpoint saved: thread_id=email-001, node=classify_email, classification=support")
print()

# Second run: same config → LangGraph reloads the checkpoint and continues
# result = graph.invoke(initial_state, config=graph_run_config)  # real call

print("Run 2: graph.invoke() called with same thread_id=email-001")
print("  Checkpoint found. Resuming from last completed node: classify_email")
print("  Skipping classify_email (already done).")
print("  Executing compose_reply...")
print("  SendMessageResult(message_id='msg_r7k2p9', thread_id='thr_newx01', status='sent')")
print()
print("Graph complete. No duplicate LLM classification call.")
print("compose_reply ran exactly once despite the crash.")
print()
print("Config used:")
print(f"  config = {graph_run_config}")
print("  graph.invoke(state, config=config)  # same config on retry = resume from checkpoint")

=== Checkpointer: resuming after crash ===

Run 1: classify_email completed. Crash before compose_reply.
  Checkpoint saved: thread_id=email-001, node=classify_email, classification=support

Run 2: graph.invoke() called with same thread_id=email-001
  Checkpoint found. Resuming from last completed node: classify_email
  Skipping classify_email (already done).
  Executing compose_reply...
  SendMessageResult(message_id='msg_r7k2p9', thread_id='thr_newx01', status='sent')

Graph complete. No duplicate LLM classification call.
compose_reply ran exactly once despite the crash.

Config used:
  config = {'configurable': {'thread_id': 'email-001'}}
  graph.invoke(state, config=config)  # same config on retry = resume from checkpoint


## Production Pattern: Monitor Reply Rates with `delivery.metrics()`

After deploying the triage graph, track whether replies are reaching inboxes. A drop in `delivery_rate` often signals a domain reputation issue, bounce spike, or misconfigured DKIM record — all fixable if caught early.

Run `client.delivery.metrics()` on a schedule (e.g., daily cron) and alert if `delivery_rate` drops below your SLA threshold or if `bounce_rate` exceeds 2%.

In [13]:
# Schedule this in a daily cron job or monitoring lambda

def check_delivery_health(
    inbox_id: str = None,
    min_delivery_rate: float = 0.95,
    max_bounce_rate: float = 0.02,
) -> dict:
    """Fetch delivery metrics and return a health assessment.

    Args:
        inbox_id: Scope to a specific inbox, or None for account-wide.
        min_delivery_rate: Alert threshold (default 95%).
        max_bounce_rate: Alert threshold (default 2%).

    Returns:
        Dict with 'healthy' bool and 'alerts' list.
    """
    metrics = client.delivery.metrics(inbox_id=inbox_id, period="7d")

    alerts = []
    if metrics.delivery_rate < min_delivery_rate:
        alerts.append(
            f"Low delivery rate: {metrics.delivery_rate:.1%} (threshold {min_delivery_rate:.0%})"
        )
    if metrics.bounce_rate > max_bounce_rate:
        alerts.append(
            f"High bounce rate: {metrics.bounce_rate:.1%} (threshold {max_bounce_rate:.0%})"
        )

    return {"healthy": len(alerts) == 0, "alerts": alerts, "metrics": metrics}


# Simulated output
print("=== Delivery metrics for last 7 days ===")
print()
print("DeliveryMetrics(sent=1247, delivered=1198, bounced=23, complained=3, delivery_rate=0.961, bounce_rate=0.018, period='7d')")
print()
print("Parsed:")
print("  sent:          1247")
print("  delivered:     1198")
print("  bounced:       23")
print("  complained:    3")
print("  delivery_rate: 96.1%  [OK - above 95% threshold]")
print("  bounce_rate:   1.8%   [OK - below 2% threshold]")
print()
print("Health check: PASS")

=== Delivery metrics for last 7 days ===

DeliveryMetrics(sent=1247, delivered=1198, bounced=23, complained=3, delivery_rate=0.961, bounce_rate=0.018, period='7d')

Parsed:
  sent:          1247
  delivered:     1198
  bounced:       23
  complained:    3
  delivery_rate: 96.1%  [OK - above 95% threshold]
  bounce_rate:   1.8%   [OK - below 2% threshold]

Health check: PASS


## Summary

LangGraph enables email triage patterns that are difficult or impossible with plain LangChain Chains:

| Capability | LangChain Chain | LangGraph |
|---|---|---|
| Conditional routing | Nested if/else in Python | Declarative `add_conditional_edges` |
| State persistence across restarts | Not built-in | `MemorySaver` / `SqliteSaver` checkpointer |
| Resume after crash | Re-run from start | Resume from last completed node |
| Thread continuity | Manual dict passing | `EmailState.thread_id` flows through all nodes |
| SMS escalation path | Separate script | `escalate_via_sms` node in the same graph |

## Next Steps

- **[async_streaming.ipynb](./async_streaming.ipynb)** — AsyncCommuneClient, concurrent processing with asyncio.gather()
- **[sms_email_combined.ipynb](./sms_email_combined.ipynb)** — multi-channel urgency routing
- **[crewai_02_production_patterns.ipynb](./crewai_02_production_patterns.ipynb)** — retry, dead-letter, idempotency
- **Commune docs:** https://commune.email/docs